# Pydantic AI weather-sub agent diagnostics

This notebook intentionally ignores the project prompts. 

the agent should take a sequence of locations and search call for weather provider
than it should return it with small summery


In [7]:
import json
import asyncio
import os
from pathlib import Path
from pprint import pprint

from pydantic import BaseModel, Field
from pydantic_ai import Agent

from capabilities.search import SearchCapability
from capabilities.weather import WeatherCapability
from capabilities.weather import default_weather_fn
from core.models import StargazingSpot, WeatherReport



async def run_with_timeout(label: str, awaitable, timeout: float = 45):
    print(f"START: {label}")
    result = await asyncio.wait_for(awaitable, timeout=timeout)
    print(f"DONE: {label}")
    return result


def load_dotenv(path: str = ".env") -> None:
    env_path = Path(path)
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value


load_dotenv()

MODEL = os.environ["LAZY_STELLAR_MODEL"]

print(f"MODEL: {MODEL}")
for key in ["OPENROUTER_API_KEY", "OPENAI_API_KEY", "TAVILY_API_KEY"]:
    print(f"{key}: {'set' if os.getenv(key) else 'missing'}")


MODEL: openrouter:google/gemini-3.1-flash-lite
OPENROUTER_API_KEY: set
OPENAI_API_KEY: missing
TAVILY_API_KEY: set


In [ ]:
class WeatherReport(BaseModel):
    summary: str = Field(description="Geneeric summery for the rigion") 
    spots: list[WeatherReport] = Field(defaut_factory=True)
    note: str = None #descri}ption="Geneeric summery for the rigion"

In [11]:
test_fake_reguest = {'spots': [{'accessibility': 'Accessible via Transilien line R from Gare de '
                             'Lyon (approx. 40-50 min). A short walk or '
                             'shuttle is required to reach the forest edges '
                             'from the station.',
            'additional_info': None,
            'bortle_class': 4,
            'description': 'A vast, historic forest region southeast of Paris. '
                           'Its significant distance from the city center '
                           'provides a much darker sky than the urban core, '
                           'making it a popular choice for astronomy '
                           'enthusiasts seeking better visibility.',
            'latitude': 48.4286,
            'longitude': 2.7001,
            'name': 'Fontainebleau Forest',
            'safety_assessment': 'Generally safe, but requires basic '
                                 'wilderness awareness (bring a flashlight, '
                                 'stay on trails, and dress for night '
                                 'temperatures). Avoid heavily wooded areas '
                                 'alone at night.',
            'seasonal_nature_risks': None,
            'source': 'General travel and astronomy resources for the '
                      'Île-de-France region.',
            'time_of_discovery': None},
           {'accessibility': 'Easily accessible via Metro Line 1 (Château de '
                             'Vincennes) or RER A (Fontenay-sous-Bois or '
                             'Vincennes stations).',
            'additional_info': None,
            'bortle_class': 7,
            'description': 'The largest public park in Paris, located on the '
                           'eastern edge of the city. While still affected by '
                           'city light pollution, its vast open spaces offer a '
                           'better vantage point than the dense city streets '
                           'for observing major constellations and planets.',
            'latitude': 48.8315,
            'longitude': 2.4414,
            'name': 'Bois de Vincennes',
            'safety_assessment': 'Public park; generally safe, but best to '
                                 'stay near well-lit pathways or within groups '
                                 "at night. The park's openness provides good "
                                 'visibility of surroundings.',
            'seasonal_nature_risks': None,
            'source': 'Urban planning and local Paris activity guides.',
            'time_of_discovery': None},
           {'accessibility': 'Accessible via Transilien Line N from Gare '
                             'Montparnasse (approx. 35-50 min). The station is '
                             'close to the outskirts of the forest.',
            'additional_info': None,
            'bortle_class': 4,
            'description': 'A large forest southwest of Paris known for its '
                           'significantly reduced light pollution compared to '
                           'the city. It provides a quiet, natural environment '
                           'suitable for stargazing once you move away from '
                           'the town center.',
            'latitude': 48.6432,
            'longitude': 1.8341,
            'name': 'Rambouillet Forest',
            'safety_assessment': 'Generally safe. As with all forest '
                                 'locations, it is recommended to remain on '
                                 'marked trails and maintain situational '
                                 'awareness, especially when accessing the '
                                 'site after dark.',
            'seasonal_nature_risks': None,}]
}

# 0 - just testing plain call

In [ ]:
weather_resolver_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    capabilities=[WeatherCapability()]
)

result = await weather_resolver_agent.run("Hi how are you?")

pprint(result)

In [ ]:
type(result)

## 1. Sub - agent assembling and test
If this hangs or fails, the problem is model/provider configuration, not search or structured output.


In [ ]:
weather_resolver_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    instructions="Using tool with 7timer API, get the weather report today's night for given locations, return the structured output, as list[WeatherReport], for each place. if there is som trubles with fetching weather, add note ",
    capabilities=[WeatherCapability()]
)

plain_result = await weather_resolver_agent.run("how what about the weather in Paris?, if there is some troubles with fetching weather, try using latitude: 48.4286," \
"longitude: 2.7001,   tell me how many times you will try to fetch the API - and what codes it will return")


print("OUTPUT:")
print(plain_result.output)
print("\nUSAGE:")
print(plain_result.usage)


## 2. Real search tool, plain text output

If step 1 works but this fails, inspect whether the model called `search_spots`, whether Tavily/DuckDuckGo failed, or whether the tool result came back as an error string.


In [ ]:
search_text_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    capabilities=[SearchCapability()],
    instructions=(
        "You are a stargazing assistant. For location-specific recommendations, "
        "call search_spots exactly once before answering. Then summarize the returned spots. "
        "If the tool returns an error string, report that exact failure briefly."
    ),
)

search_text_result = await run_with_timeout(
    "search tool + plain text output",
    search_text_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("OUTPUT:")
print(search_text_result.output)
print("\nUSAGE:")
print(search_text_result.usage)
print("\nMESSAGES:")
for message in search_text_result.all_messages():
    print(type(message).__name__, message)


## 2a. Real search tool, JSON text output

This tests whether the model can use the search tool and emit JSON text when Pydantic AI does not force the final `output_type` tool.


In [ ]:
from pydantic_ai.capabilities import WebSearch
from pydantic_ai.capabilities import Thinking


json_text_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    # capabilities=[SearchCapability()],
    capabilities=[WebSearch(), Thinking("high")],
    instructions=(
        "You are a stargazing assistant. You must call search_spots exactly once before final output. "
        "Then return ONLY valid JSON text with keys: spots_found, summary, spots. "
        "spots must be an array of objects with name, latitude, longitude, source, description, accessibility, safety_assessment, and bortle_class. "
        "If search_spots returns an error string, return {\"spots_found\": 0, \"summary\": <error>, \"spots\": []}. "
        "Do not wrap the JSON in markdown. Do not invent spots that were not returned by the tool."
    ),
)

json_text_result = await run_with_timeout(
    "search tool + JSON text output",
    json_text_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("RAW OUTPUT:")
print(json_text_result.output)
print("\nPARSED JSON:")
pprint(json.loads(json_text_result.output))
print("\nUSAGE:")
print(json_text_result.usage)
print("\nMESSAGES:")
for message in json_text_result.all_messages():
    print(type(message).__name__, message)


## 3. Real search tool, structured Pydantic output

If steps 1 and 2 work but this fails, the problem is structured output/tool interaction. This cell forces a `SpotSearchReport` final result.


In [ ]:
structured_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    output_type=SpotSearchReport,
    capabilities=[WebSearch(), Thinking(effort=True)],
    instructions=(
        "You are a stargazing assistant. You must call search_spots exactly once before final output. "
        "Use the tool result to fill SpotSearchReport. "
        "If search_spots returns an error string, return spots_found=0, spots=[], and put the error in summary. "
        "Do not invent spots that were not returned by the tool."
        "before returning rechack all fields for correctness and consistency, and if you find any issues, use the search tool again"
        "You can use search tool max 2 times for each spot"
    ),
)

structured_result = await run_with_timeout(
    "search tool + structured Pydantic output",
    structured_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("PYDANTIC OUTPUT:")
pprint(structured_result.output.model_dump())
print("\nJSON:")
print(structured_result.output.model_dump_json(indent=2))
print("\nUSAGE:")
print(structured_result.usage)
print("\nMESSAGES:")
for message in structured_result.all_messages():
    print(type(message).__name__, message)


In [ ]:
# checking Api function

In [10]:
result = default_weather_fn(longitude=2.7001, latitude=48.8566)

In [11]:
pprint(result)

WeatherReport(name='Astro weather at (48.8566, 2.7001)',
              latitude=48.8566,
              longitude=2.7001,
              cloud_cover={'00:00': 1,
                           '03:00': 1,
                           '06:00': 1,
                           '09:00': 1,
                           '21:00': 1},
              transparency={'00:00': 3,
                            '03:00': 4,
                            '06:00': 3,
                            '09:00': 3,
                            '21:00': 3},
              seeing={'00:00': 5,
                      '03:00': 5,
                      '06:00': 3,
                      '09:00': 3,
                      '21:00': 5},
              wind_speed={'00:00': 2,
                          '03:00': 2,
                          '06:00': 2,
                          '09:00': 2,
                          '21:00': 2},
              wind_direction={'00:00': 'NE',
                              '03:00': 'N',
                              '

In [ ]:
print(2+2)


In [8]:
# simple Api call

import httpx


url = "http://www.7timer.info/bin/api.pl?lon=113.17&lat=23.09&product=astro&output=json"

response = httpx.get(url)

In [9]:
print(response.text)

{
	"product" : "astro" ,
	"init" : "2026052406" ,
	"dataseries" : [
	{
		"timepoint" : 3,
		"cloudcover" : 9,
		"seeing" : 6,
		"transparency" : 3,
		"lifted_index" : -4,
		"rh2m" : 7,
		"wind10m" : {
			"direction" : "S",
			"speed" : 3
		},
		"temp2m" : 34,
		"prec_type" : "none"
	},
	{
		"timepoint" : 6,
		"cloudcover" : 9,
		"seeing" : 6,
		"transparency" : 4,
		"lifted_index" : -4,
		"rh2m" : 10,
		"wind10m" : {
			"direction" : "S",
			"speed" : 3
		},
		"temp2m" : 30,
		"prec_type" : "none"
	},
	{
		"timepoint" : 9,
		"cloudcover" : 9,
		"seeing" : 6,
		"transparency" : 5,
		"lifted_index" : -4,
		"rh2m" : 10,
		"wind10m" : {
			"direction" : "S",
			"speed" : 3
		},
		"temp2m" : 29,
		"prec_type" : "none"
	},
	{
		"timepoint" : 12,
		"cloudcover" : 9,
		"seeing" : 6,
		"transparency" : 5,
		"lifted_index" : -4,
		"rh2m" : 12,
		"wind10m" : {
			"direction" : "S",
			"speed" : 3
		},
		"temp2m" : 28,
		"prec_type" : "none"
	},
	{
		"timepoint" : 15,
		"cloudcover" : 8,
		"seeing